Managing the Conversation History 

In [1]:
# One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unboounded and potentially overflow the context window of the llm. Therefore, it is important to add a step that limits the size of the messages you are passing in.

In [2]:
# trim msgs helper function  helps reduce how many msgs we're sending to the model. The trimmer allows us to speicfy how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [6]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="groq/compound",groq_api_key=groq_api_key)
llm

c:\Generative AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x0000016F6948B500>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016F6739FCB0>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [17]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens = 70,
    strategy = 'last',
    token_counter =llm,
    include_system = True,
    allow_partial = False,
    start_on = "human"
)


In [9]:
from langchain_core.messages import HumanMessage, AIMessage

In [30]:
messages =[SystemMessage(content="You are a helpful assistant."),
           HumanMessage(content="Tell me a joke."),
           AIMessage(content="Why did the scarecrow win an award? Because he was outstanding in his field!"),
           HumanMessage(content="Tell me another joke."),
           AIMessage(content="Why don't scientists trust atoms? Because they make up everything!"),
           HumanMessage(content="Tell me one more joke."),
           AIMessage(content="Why did the math book look sad? Because it had too many problems!")]

In [31]:
trimmer.invoke(messages)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Tell me another joke.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Why don't scientists trust atoms? Because they make up everything!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Tell me one more joke.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Why did the math book look sad? Because it had too many problems!', additional_kwargs={}, response_metadata={})]

In [20]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
("system",
        "You are a helpful assistant. Answer all the questions to the best of your ability. "
        "Always respond in this language: {language}."),
MessagesPlaceholder(variable_name="messages")
])

In [24]:
# how to pass trimmer in chain
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain = RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer) | prompt | llm

chain.invoke({"messages":messages + [HumanMessage(content="Can you tell me a riddle?")], "language":"Hindi"})



AIMessage(content='**पहेली:**  \n*मेरे पास सिर है, पूँछ है, लेकिन शरीर नहीं। मैं क्या हूँ?*  \n\n**विचार‑प्रक्रिया (Reasoning):**  \n- “सिर” और “पूँछ” शब्द अक्सर दो अलग‑अलग पक्षों को दर्शाते हैं।  \n- कई वस्तुओं में दोनों पक्ष होते हैं, पर उनका कोई “शरीर” नहीं होता।  \n- सिक्के के एक पक्ष पर अक्सर एक चित्र (जैसे राष्ट्रपति) होता है, जिसे “सिर” कहा जाता है, और दूसरे पक्ष पर अक्सर पशु या प्रतीक (जैसे “पूँछ”) होता है।  \n- सिक्के का कोई ठोस शरीर नहीं होता; वह केवल दो सतहों (सिर‑पूँछ) से बना होता है।\n\n**उत्तर:** *सिक्का* (एक कॉइन)  \n\nयदि आप और पहेलियाँ या riddles चाहते हैं, तो बताइए!', additional_kwargs={'reasoning_content': '\nHere\'s a riddle for you: \nI have a head, a tail, but no body. What am I?\n\n<tool>python(print("Think you know the answer?"))</tool>\n<output>Think you know the answer?\n</output>\n\n\nThe answer is: A coin! \n\nI have a head on one side and a tail on the other, but I don\'t have a body. \n\nLet me know if you want another riddle!', 'executed_tools': [{'argume

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [25]:
store = {}

In [27]:
def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]= ChatMessageHistory()
    return store[session_id]

In [28]:
# let us wrap this in the msg history class for easier reuse
with_message_history = RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages",)
config = {"configurable":{"session_id":"chat5"}}

In [32]:
response = with_message_history.invoke({
    "messages": messages + [HumanMessage(content="What is my name?")],
    "language":"English",
},config=config,)
response.content

'I’m sorry, but I don’t have any information about your name from our conversation so far. If you’d like me to address you by name, just let me know what it is!'